## 1. Connect and prepare folders
Connects to PostgreSQL and creates the `data/` folder where generated files are saved. `data/` is in `.gitignore`, so it is never pushed to GitHub.

In [1]:
# --- Imports ---
import os                                   # file paths and environment variables
from dotenv import load_dotenv              # reads settings from the .env file
import psycopg                              # PostgreSQL driver

# --- Load the database URL and connect (same pattern as notebook 01) ---
load_dotenv(os.path.expanduser("~/bank-rag-guard/.env"))
conn = psycopg.connect(os.environ["DATABASE_URL"], autocommit=True)

# --- Create the data folder if it does not exist yet ---
DATA_DIR = os.path.expanduser("~/bank-rag-guard/data")
os.makedirs(DATA_DIR, exist_ok=True)
print("Connected. Data folder:", DATA_DIR)

Connected. Data folder: /home/linux1/bank-rag-guard/data


## 2. Helpers that generate valid-looking PII
- `luhn_check_digit`: computes the final check digit of an Emirates ID
- `make_emirates_id`: format `784-YYYY-XXXXXXX-Z`
- `make_uae_iban`: 23-character IBAN starting with `AE`, with correct mod-97 check digits
- `to_arabic_digits`: converts 0-9 to Arabic-Indic digits, to test the masker on Arabic-written numbers

In [2]:
import random
import re

SEED = 42            # fixed seed so every run generates the same data
random.seed(SEED)

def luhn_check_digit(digits: str) -> str:
    """Return the Luhn check digit for a string of digits."""
    total = 0
    # Walk from the rightmost digit; double every second digit starting with the first
    for i, ch in enumerate(reversed(digits)):
        d = int(ch)
        if i % 2 == 0:
            d *= 2
            if d > 9:
                d -= 9               # same as adding the two digits of the product
        total += d
    return str((10 - total % 10) % 10)

def make_emirates_id() -> str:
    """Build an ID in the format 784-YYYY-XXXXXXX-Z."""
    year = random.randint(1950, 2005)               # birth year part
    serial = f"{random.randint(0, 9_999_999):07d}"  # 7 random digits
    check = luhn_check_digit("784" + str(year) + serial)
    return f"784-{year}-{serial}-{check}"

def make_uae_iban() -> str:
    """Build a 23-character UAE IBAN: AE + 2 check digits + 3-digit bank + 16-digit account."""
    bank = f"{random.randint(0, 999):03d}"
    account = f"{random.randint(0, 10**16 - 1):016d}"
    bban = bank + account
    # IBAN check digits: append "AE00" as numbers (A=10, E=14), then 98 - (value mod 97)
    check = 98 - int(bban + "101400") % 97
    return f"AE{check:02d}{bban}"

def make_phone() -> str:
    """UAE mobile number, e.g. +971501234567."""
    return "+9715" + f"{random.randint(0, 99_999_999):08d}"

def to_arabic_digits(text: str) -> str:
    """Replace 0-9 with Arabic-Indic digits (٠-٩)."""
    return text.translate(str.maketrans("0123456789", "٠١٢٣٤٥٦٧٨٩"))

# --- Quick self-test of the generators ---
sample_eid, sample_iban = make_emirates_id(), make_uae_iban()
assert re.fullmatch(r"784-\d{4}-\d{7}-\d", sample_eid)
assert len(sample_iban) == 23 and sample_iban.startswith("AE")
print(sample_eid, sample_iban, make_phone())

784-1990-1867825-2 AE110254954488479039739 +971532868828


## 3. Generate synthetic customer records
Creates 300 fake customers with English or Arabic names, plus a short case note that embeds their Emirates ID, IBAN and phone number. About a third of the Arabic notes write numbers in Arabic-Indic digits. The raw fields are kept as **ground truth** so we can later measure how well the masker catches them. Each record gets a clearance tier (1 = most common, 3 = most restricted).

In [3]:
import pandas as pd
from faker import Faker

Faker.seed(SEED)
fake_en = Faker("en_US")       # English names
fake_ar = Faker("ar_AA")       # Arabic names (Arabic script)

# --- Case-note templates; {placeholders} are filled with the fake PII ---
TEMPLATE_EN = ("Customer {name}, Emirates ID {eid}, IBAN {iban}, phone {phone}, "
               "reported a disputed card transaction and requested a review.")
TEMPLATE_AR = ("العميل {name}، رقم الهوية الإماراتية {eid}، رقم الآيبان {iban}، "
               "الهاتف {phone}، قدّم شكوى بخصوص عملية بطاقة مشكوك فيها.")

rows = []
for i in range(300):
    lang = random.choice(["en", "ar"])
    name = fake_en.name() if lang == "en" else fake_ar.name()
    eid, iban, phone = make_emirates_id(), make_uae_iban(), make_phone()

    # Fill the template; for ~1/3 of Arabic notes, write the numbers in Arabic digits
    text = (TEMPLATE_EN if lang == "en" else TEMPLATE_AR).format(
        name=name, eid=eid, iban=iban, phone=phone)
    if lang == "ar" and random.random() < 0.33:
        text = to_arabic_digits(text)

    rows.append({
        "customer_id": f"CUST{i:04d}",
        "language": lang,
        "name": name, "emirates_id": eid, "iban": iban, "phone": phone,  # ground truth
        "text": text,
        # Tier mix: 50% tier 1, 30% tier 2, 20% tier 3
        "tier": random.choices([1, 2, 3], weights=[50, 30, 20])[0],
    })

customers = pd.DataFrame(rows)
customers.to_csv(f"{DATA_DIR}/customers.csv", index=False)   # saved for later notebooks
print(customers["language"].value_counts().to_dict(), customers["tier"].value_counts().to_dict())
customers.head(4)

{'ar': 152, 'en': 148} {1: 148, 2: 100, 3: 52}


,customer_id,language,name,emirates_id,iban,phone,text,tier
0,CUST0000,en,Allison Hill,784-1958-1719583-7,AE646921566149281915956,+971579254563,"Customer Allison Hill, Emirates ID 784-1958-17...",1
1,CUST0001,en,Noah Rhodes,784-1955-3668136-4,AE952387557396729776374,+971529587039,"Customer Noah Rhodes, Emirates ID 784-1955-366...",1
2,CUST0002,ar,محمود الألجاوي,784-2001-0109031-2,AE097772876166965439421,+971593702683,العميل محمود الألجاوي، رقم الهوية الإماراتية 7...,1
3,CUST0003,en,Daniel Wagner,784-1998-5647119-0,AE401046844159633535249,+971512981052,"Customer Daniel Wagner, Emirates ID 784-1998-5...",1


## 4. Create demo users
Adds one account per role: a teller (tier 1), a compliance officer (tier 3) and an auditor. These are used to prove that the same question gives different answers depending on clearance.

In [4]:
# --- (username, role, tier) for each demo account ---
demo_users = [
    ("teller1",     "teller",     1),   # branch staff: lowest clearance
    ("compliance1", "compliance", 3),   # compliance officer: highest clearance
    ("auditor1",    "auditor",    1),   # auditor: checks logs, not documents
]

for user in demo_users:
    # ON CONFLICT DO NOTHING makes this safe to re-run
    conn.execute(
        "INSERT INTO users (username, role, tier) VALUES (%s, %s, %s) "
        "ON CONFLICT (username) DO NOTHING", user)

print(conn.execute("SELECT * FROM users ORDER BY username").fetchall())

[('auditor1', 'auditor', 1), ('compliance1', 'compliance', 3), ('teller1', 'teller', 1)]


## 5. Check internet access
Tests whether the server can reach Hugging Face (dataset downloads) and the Gemini and Groq APIs. Any HTTP status number means the site is reachable; FAILED means it is blocked.

In [5]:
import requests

# --- Sites we need: dataset hub and the two LLM providers ---
sites = [
    "https://huggingface.co",
    "https://generativelanguage.googleapis.com",
    "https://api.groq.com",
]

for url in sites:
    try:
        r = requests.get(url, timeout=10)
        print(f"{url} -> reachable (HTTP {r.status_code})")   # 200/401/403/404 are all fine here
    except Exception as e:
        print(f"{url} -> FAILED ({type(e).__name__})")

https://huggingface.co -> reachable (HTTP 200)
https://generativelanguage.googleapis.com -> reachable (HTTP 404)
https://api.groq.com -> reachable (HTTP 200)
